In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv("Dataset/predictive_maintenance_dataset_v2.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.shape 

In [ ]:
target_counts = df['failure'].value_counts()

label_map = {0: "No Failure",1: "Failure"}

labels = [label_map[i] for i in target_counts.index]
sizes = target_counts.values

plt.figure(figsize=(5,5))
plt.pie( sizes, labels=labels, autopct='%1.1f%%', startangle=45, explode=[0.05]*len(labels))
plt.title("Failure Class Distribution")
plt.axis('equal')
plt.tight_layout()
plt.show()

# Correlation Matrix

In [ ]:
plt.figure(figsize = (15,10))
sns.heatmap(df.corr(), annot = True, cmap="Blues")
plt.title('Correlation Matrix')
plt.show()

In [ ]:
imputer = SimpleImputer(strategy="median")

numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = imputer.fit_transform(df[numeric_cols])


# Feature Engineering

In [ ]:
def aggregate_features(df):

    # separate target
    target = df.groupby("vehicle_id")["failure"].last()

    # drop target before aggregation
    df_features = df.drop(columns=["failure"])

    grouped = df_features.groupby("vehicle_id")

    mean_features = grouped.mean().add_suffix("_mean")
    std_features = grouped.std().add_suffix("_std")
    last_features = grouped.last().add_suffix("_last")

    features = pd.concat(
        [mean_features, std_features, last_features],
        axis=1
    )

    features["failure"] = target

    features.reset_index(inplace=True)

    return features

In [ ]:
df = aggregate_features(df)

print(df.shape)

In [ ]:
print(df.columns.tolist())

# Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include="object").columns

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
drop_cols = ["vehicle_id",
     "time_step_mean","time_step_std", "time_step_last",
    "event_occurred_mean","event_occurred_std","event_occurred_last",
    "event_time_mean","event_time_std","event_time_last",
    "time_to_event_mean","time_to_event_std","time_to_event_last",
    "is_censored_mean","is_censored_std","is_censored_last"
]

X = df.drop(columns=drop_cols + ["failure"])
y = df["failure"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

# Feature Selection 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier( n_estimators=300, random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

importances = rf.feature_importances_

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
})

feature_importance = feature_importance.sort_values(by="importance", ascending=False)

selected_features = feature_importance.head(20)["feature"]

print("Selected Features:", list(selected_features))

X_train_fs = X_train[selected_features]
X_test_fs = X_test[selected_features]

In [ ]:
X_train_fs = pd.DataFrame(X_train_fs, columns=selected_features)
X_test_fs = pd.DataFrame(X_test_fs, columns=selected_features)

# SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train_fs, y_train)

joblib.dump(X_train, "Models/lime_training_data.pkl")

print("Before SMOTE:", X_train.shape, y_train.shape)
print("After SMOTE:", X_train_res.shape, y_train_res.shape)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

ML_Model = []
accuracy = []
precision = []
recall = []
f1score = []
fp = []
fn =[]
tc = []

def storeResults(model, a, b, c, d, e, f, g):
    ML_Model.append(model) 
    accuracy.append(round(a, 3)) 
    precision.append(round(b, 3))
    recall.append(round(c, 3))
    f1score.append(round(d, 3))
    fp.append(round(e, 3))
    fn.append(round(f, 3))
    tc.append(round(g, 3))
    

# RandomForest

In [ ]:


rf = RandomForestClassifier(max_depth=5, n_estimators=50, random_state=42)
rf.fit(X_train_res, y_train_res)

y_pred = rf.predict(X_test_fs)

# Standard metrics
rf_acc = accuracy_score(y_test, y_pred)
rf_prec = precision_score(y_test, y_pred, average='weighted')
rf_rec = recall_score(y_test, y_pred, average='weighted')
rf_f1 = f1_score(y_test, y_pred, average='weighted')

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')


TN, rf_FP, rf_FN, TP = cm.ravel()

cost_fp = 1     # cost of false alarm
cost_fn = 10    # cost of missed failure

rf_total_cost = (rf_FP * cost_fp) + (rf_FN * cost_fn)

print("\nCost-Based Evaluation")
print("----------------------")
print("False Positives:", rf_FP)
print("False Negatives:", rf_FN)
print("Total Cost:", rf_total_cost)

In [ ]:
storeResults("RandomForest", rf_acc, rf_prec, rf_rec, rf_f1, rf_FP, rf_FN, rf_total_cost)

# DecisionTree

In [ ]:

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train_res, y_train_res)

y_pred = dt.predict(X_test_fs)

# Standard metrics
dt_acc = accuracy_score(y_test, y_pred)
dt_prec = precision_score(y_test, y_pred, average='weighted')
dt_rec = recall_score(y_test, y_pred, average='weighted')
dt_f1 = f1_score(y_test, y_pred, average='weighted')

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')


TN, dt_FP, dt_FN, TP = cm.ravel()

cost_fp = 1     # cost of false alarm
cost_fn = 10    # cost of missed failure

dt_total_cost = (dt_FP * cost_fp) + (dt_FN * cost_fn)

print("\nCost-Based Evaluation")
print("----------------------")
print("False Positives:", dt_FP)
print("False Negatives:", dt_FN)
print("Total Cost:", dt_total_cost)

In [ ]:
storeResults("DecisionTree", dt_acc, dt_prec, dt_rec, dt_f1, dt_FP, dt_FN, dt_total_cost)

# GradientBoosting

In [ ]:

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
gb.fit(X_train_res, y_train_res)

y_pred = gb.predict(X_test_fs)

# Standard metrics
gb_acc = accuracy_score(y_test, y_pred)
gb_prec = precision_score(y_test, y_pred, average='weighted')
gb_rec = recall_score(y_test, y_pred, average='weighted')
gb_f1 = f1_score(y_test, y_pred, average='weighted')

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')


TN, gb_FP, gb_FN, TP = cm.ravel()

cost_fp = 1     # cost of false alarm
cost_fn = 10    # cost of missed failure

gb_total_cost = (gb_FP * cost_fp) + (gb_FN * cost_fn)

print("\nCost-Based Evaluation")
print("----------------------")
print("False Positives:", gb_FP)
print("False Negatives:", gb_FN)
print("Total Cost:", gb_total_cost)

In [ ]:
storeResults("GradientBoosting", gb_acc, gb_prec, gb_rec, gb_f1, gb_FP, gb_FN, gb_total_cost)

# XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train_res, y_train_res)

y_pred = xgb.predict(X_test_fs)

# Standard metrics
xgb_acc = accuracy_score(y_test, y_pred)
xgb_prec = precision_score(y_test, y_pred, average='weighted')
xgb_rec = recall_score(y_test, y_pred, average='weighted')
xgb_f1 = f1_score(y_test, y_pred, average='weighted')

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')


# COST-BASED METRIC

TN, xgb_FP, xgb_FN, TP = cm.ravel()

cost_fp = 1     # cost of false alarm
cost_fn = 10    # cost of missed failure

xgb_total_cost = (xgb_FP * cost_fp) + (xgb_FN * cost_fn)

print("\nCost-Based Evaluation")
print("----------------------")
print("False Positives:", xgb_FP)
print("False Negatives:", xgb_FN)
print("Total Cost:", xgb_total_cost)

In [ ]:
storeResults("XGBoost", xgb_acc, xgb_prec, xgb_rec, xgb_f1, xgb_FP, xgb_FN, xgb_total_cost)

# LogisticRegression

In [ ]:
lr = LogisticRegression(C=1, max_iter=10, random_state=42)

lr.fit(X_train_res, y_train_res)

y_pred = lr.predict(X_test_fs)

# Standard metrics
lr_acc = accuracy_score(y_test, y_pred)
lr_prec = precision_score(y_test, y_pred, average='weighted')
lr_rec = recall_score(y_test, y_pred, average='weighted')
lr_f1 = f1_score(y_test, y_pred, average='weighted')

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')

# COST-BASED METRIC

TN, lr_FP, lr_FN, TP = cm.ravel()

cost_fp = 1 
cost_fn = 10 

lr_total_cost = (lr_FP * cost_fp) + (lr_FN * cost_fn)

print("\nCost-Based Evaluation")
print("----------------------")
print("False Positives:", lr_FP)
print("False Negatives:", lr_FN)
print("Total Cost:", lr_total_cost)

In [ ]:
storeResults("LogisticRegression", lr_acc, lr_prec, lr_rec, lr_f1, lr_FP, lr_FN, lr_total_cost)

# Voting Classifier

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=7,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1,
    eval_metric='logloss',
    random_state=42
)

gb = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=4,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42
)

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)
lr = LogisticRegression( max_iter=1000, random_state=42)
voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', xgb),
        ('gb', gb),
        ('lgbm', lgbm),
        ('LR', lr)
    ],
    voting='soft',
    n_jobs=-1
)

voting.fit(X_train_res, y_train_res)

y_pred = voting.predict(X_test_fs)
y_prob = voting.predict_proba(X_test_fs)[:,1]


vote_acc = accuracy_score(y_test, y_pred)
vote_prec = precision_score(y_test, y_pred, average='weighted')
vote_rec = recall_score(y_test, y_pred, average='weighted')
vote_f1 = f1_score(y_test, y_pred, average='weighted')

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["No Failure", "Failure"]))


cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues')


TN, vote_FP, vote_FN, TP = cm.ravel()

cost_fp = 1     # false alarm cost
cost_fn = 10    # missed failure cost

vote_total_cost = (vote_FP * cost_fp) + (vote_FN * cost_fn)

print("\nCost-Based Evaluation")
print("----------------------")
print("False Positives:", vote_FP)
print("False Negatives:", vote_FN)
print("Total Cost:", vote_total_cost)

In [ ]:
storeResults("Voting Classifier", vote_acc, vote_prec, vote_rec, vote_f1, vote_FP, vote_FN, vote_total_cost)

In [ ]:
joblib.dump(voting, "Models/Voting_model.sav")

In [ ]:
result = pd.DataFrame({ 'ML Model' : ML_Model,
                        'Accuracy' : accuracy,
                        'Precision': precision,
                        'Recall'   : recall,
                        'F1-Score': f1score,
                        'False-Positive': fp,
                       'False-Negative': fn,
                       'Total Cost': tc,
                        
                      })

# Comparison Table

In [ ]:
result

# Comparison Graphs

In [ ]:
palettes = {
    'Accuracy': 'Blues',
    'Precision': 'Greens',
    'Recall': 'Purples',
    'F1-Score': 'Oranges',
    'False-Positive': 'pink',
    'False-Negative' : 'Greens',
    'Total Cost' : 'Reds'
}

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'False-Positive', 'False-Negative', 'Total Cost']:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=result, x='ML Model', y=metric, palette=palettes[metric])
    plt.title(f'{metric} by Model')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# XAI

In [ ]:
import joblib

model = joblib.load("Models/Voting_model.sav")

# XAI - LIME

In [ ]:
import numpy as np
from lime import lime_tabular

feature_names = X.columns.tolist()

X_train_np = X_train_res.to_numpy() if not isinstance(X_train_res, np.ndarray) else X_train_res
X_test_np  = X_test_fs.to_numpy()  if not isinstance(X_test_fs,  np.ndarray) else X_test_fs

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=feature_names,
    class_names=labels,         
    mode='classification'
)

i = 5
explanation = lime_explainer.explain_instance(
    data_row=X_test_np[i],
    predict_fn=model.predict_proba,  
    top_labels=len(labels),       
    num_features=20               
)

explanation.show_in_notebook(show_table=True, show_all=True)

In [ ]:
import plotly.graph_objects as go

label_idx = explanation.available_labels()[0]
exp_list = explanation.as_list(label=label_idx)
features, weights = zip(*exp_list)

fig = go.Figure(go.Waterfall(
    name="LIME Explanation",
    orientation="v",
    measure=["relative"] * len(weights),
    x=features,
    text=[f"{w:.2f}" for w in weights],
    y=weights,
))

fig.update_layout(title=f"LIME Waterfall Explanation (Label: {label_idx})")
fig.show()

# XAI - SHAP

In [ ]:
import shap

X_train_df = pd.DataFrame(X_train_np)

X_background = X_train_df.sample(10, random_state=42)
X_explain = X_train_df.sample(100, random_state=1)

explainer = shap.KernelExplainer(model.predict_proba, X_background)
shap_values = explainer.shap_values(X_explain, nsamples=100)

shap_class_1 = shap_values[:, :, 1]

print("SHAP shape:", shap_class_1.shape)
shap.summary_plot(shap_class_1, X_explain, feature_names=feature_names)